# Exemplary Notebook for the Pandas Skyfield Extension

## Configuration of the module

The Pandas Skyfield Extension has a configuration object `config` that can be used to set preferences for the conversion between `skyfield.units` objects like `Distance`, `Velocity` and `Angle` to `astropy` `Quantity` objects. To do so use the following code snipped:

```python
>>> import astropy.units as u
>>> import pandas_skyfield_extension as psfe

>>> # Show standard units
>>> print(f"Standard length unit:", repr(psfe.config.length_unit))
Standard length unit: Unit("km")
>>> print(f"Standard velocity unit:", repr(psfe.config.velocity_unit))
Standard velocity unit: Unit("km / s")
>>> print(f"Standard angle unit:", repr(psfe.config.angle_unit))
Standard angle unit: Unit("deg")

>>> # To change the unit just assign a new value:
>>> psfe.config.length_unit = u.m

>>> # Show new unit:
>>> print(f"Changed length unit:", repr(psfe.config.length_unit))
Changed length unit: Unit("m")
```


## Imports

In [1]:
import datetime

import astropy.units as u
import numpy as np
import pandas as pd
import pandas_skyfield_extension as psfe
import skyfield.api as sf
from skyfield import framelib, positionlib, sgp4lib, toposlib

## Defining test objects

Define two `Skyfield` satellites (`nanoff_a`, `nanoff_b`), a ground station (`berlin_gs`) and a collection of times (`times_idx`) that will be used throughout this notebook as an example.

In [2]:
# Define TLE
NanoFF_A_TLE_1: str = "1 58810U 23185T   26033.42930604  .00003749  00000-0  22676-3 0  9992"
NanoFF_A_TLE_2: str = "2 58810  97.5437 102.7707 0011522  57.3014 302.9326 15.11204711112680"

NanoFF_B_TLE_1: str = "1 58755U 23185S   26033.36314431  .00003748  00000-0  22656-3 0  9994"
NanoFF_B_TLE_2: str = "2 58755  97.5440 102.7270 0013756  53.5736 306.6760 15.11205878113507"

# Create EarthSatellites for NanoFF-A and NanoFF-B
nanoff_a: sf.EarthSatellite = sf.EarthSatellite(NanoFF_A_TLE_1, NanoFF_A_TLE_2, name="NanoFF-A")
nanoff_b: sf.EarthSatellite = sf.EarthSatellite(NanoFF_B_TLE_1, NanoFF_B_TLE_2, name="NanoFF-B")

# Create location for Berlin ground station 
berlin_gs: toposlib.GeographicPosition = sf.wgs84.latlon(52.5148, 13.3236, elevation_m=100)

# Define times of interest
start_time: datetime.datetime = datetime.datetime.fromisoformat("2026-02-02T00:00:00Z")
end_time: datetime.datetime = datetime.datetime.fromisoformat("2026-02-03T00:00:00Z")
resolution: str = "1 min"
times_idx: pd.DatetimeIndex = pd.date_range(start=start_time, end=end_time, freq=resolution)

## The Skyfield native way

While the `pandas_units_extension` module already provides support for astropy `Quantity` objects and many of the Skyfields position properties can be converted to these an early conversion can complicate alignment or calculations.

In [3]:
# Loading the default timescale
ts: sf.Timescale = sf.load.timescale()

# Convert times to Skyfield Time object
times_sf: sf.Time = ts.from_datetimes(times_idx.to_pydatetime())

# Calculate positions at the given times
nanoff_a_pos = nanoff_a.at(times_sf)
nanoff_b_pos = nanoff_b.at(times_sf)

# Extract position and velocity components for NanoFF-A as Astropy Quantities
x, y, z = nanoff_a_pos.position.to(u.km)
vx, vy, vz = nanoff_a_pos.velocity.to(u.km / u.s)

# Create a DataFrame to hold the position and velocity data
nanoff_a_df = pd.DataFrame({
        "x": x,
        "y": y,
        "z": z,
        "vx": vx,
        "vy": vy,
        "vz": vz
    },
    dtype="unit",
    index=times_idx
)
nanoff_a_df

,x,y,z,vx,vy,vz
2026-02-02 00:00:00+00:00,1330.6511750133932 km,-6747.060153182442 km,-764.3481875200815 km,-1.1593142248501966 km / s,0.6035419883019602 km / s,-7.475086675980539 km / s
2026-02-02 00:01:00+00:00,1258.258441042206 km,-6696.24976253448 km,-1210.8671184092202 km,-1.2528961729943071 km / s,1.0894823986775657 km / s,-7.403492083367096 km / s
2026-02-02 00:02:00+00:00,1180.413133140199 km,-6616.421443001309 km,-1652.1244746036548 km,-1.3410028375002463 km / s,1.5704584554992944 km / s,-7.2997701317657855 km / s
2026-02-02 00:03:00+00:00,1097.4552478728413 km,-6507.935975320599 km,-2086.206541517927 km,-1.4232557477786318 km / s,2.0443782766600576 km / s,-7.164407480149509 km / s
2026-02-02 00:04:00+00:00,1009.7466918613346 km,-6371.278595921668 km,-2511.232980290524 km,-1.4993032297870654 km / s,2.5091865365032433 km / s,-6.998029407274324 km / s
...,...,...,...,...,...,...
2026-02-02 23:56:00+00:00,947.410783644335 km,-6046.651002549271 km,-3236.7607472869067 km,-1.674094505186255 km / s,3.276616760778375 km / s,-6.62894467795052 km / s
2026-02-02 23:57:00+00:00,844.9909167385426 km,-5837.133260295946 km,-3627.198743447495 km,-1.7386692369456849 km / s,3.7047637425959743 km / s,-6.380967922029718 km / s
2026-02-02 23:58:00+00:00,738.9216486739757 km,-5602.405196433733 km,-4001.92800851328 km,-1.7956986412802531 km / s,4.116662079783719 km / s,-6.1055099940109105 km / s
2026-02-02 23:59:00+00:00,629.6629679975924 km,-5343.494802121336 km,-4359.335814597282 km,-1.8449456965957887 km / s,4.510554845960358 km / s,-5.803794162492498 km / s


## Creating a Skyfield position Series

Using this pandas extension it is possible to store Skyfield position objects natively in `pd.Series` or `pd.DataFrame` without the need to convert them to a `float` or astropy `Quantity`.

### For a single object

- Using `psfe.at(obj, times)` instead of `obj.at(times)` allows to use `sf.Time` or any of `datetime.datetime`, `pd.Timestamp` or even `str` as long it is understood by `pd.to_datetime` for the `times` argument. For everything besides `sf.Time` the default `Timescale` will be used if not provided by the `ts` `kwarg`, see `psfe.to_sf_time`.
- `psfe.at(obj, times)` creates a `pd.Series` with the calculated Skyfield position in the background using a `SkyfieldPositionDtype` and a `SkyfieldPositionExtensionArray` in the backend to provide vectorized access to the position.
- By default the returned `pd.Series` will be created with a `pd.DatetimeIndex` as `index`. This is the case as long as `index` is not specified in `kwargs` and `set_datetime_idx` is not set to `False`.

In [4]:
nanoff_a_sr: pd.Series = psfe.at(nanoff_a, times_idx)
nanoff_a_sr

2026-02-02 00:00:00+00:00    Geocentric position [ 1330.651 -6747.060  -764...
2026-02-02 00:01:00+00:00    Geocentric position [ 1258.258 -6696.250 -1210...
2026-02-02 00:02:00+00:00    Geocentric position [ 1180.413 -6616.421 -1652...
2026-02-02 00:03:00+00:00    Geocentric position [ 1097.455 -6507.936 -2086...
2026-02-02 00:04:00+00:00    Geocentric position [ 1009.747 -6371.279 -2511...
                                                   ...                        
2026-02-02 23:56:00+00:00    Geocentric position [  947.411 -6046.651 -3236...
2026-02-02 23:57:00+00:00    Geocentric position [  844.991 -5837.133 -3627...
2026-02-02 23:58:00+00:00    Geocentric position [  738.922 -5602.405 -4001...
2026-02-02 23:59:00+00:00    Geocentric position [  629.663 -5343.495 -4359...
2026-02-03 00:00:00+00:00    Geocentric position [  517.688 -5061.533 -4697...
Freq: min, Name: NanoFF-A, Length: 1441, dtype: skyfield_position[<class 'skyfield.positionlib.Geocentric'>]

### For relative position between two objects

It is possible to create a Skyfield position `Series` for any `obj` that is a `skyfield.vectorlib.VectorFunction` which includes `sf.EarthSatellite`, `skyfield.toposlib.GeographicPosition` and `skyfield.vectorlib.VectorSum`. The later is usually created by calculating the difference between two objects like in the following examples:

In [5]:
nanoff_rel_sr: pd.Series = psfe.at(nanoff_b - nanoff_a, times_idx)
nanoff_rel_sr

2026-02-02 00:00:00+00:00    ICRF position [    7.565    -3.307    30.551] ...
2026-02-02 00:01:00+00:00    ICRF position [    7.955    -5.396    30.348] ...
2026-02-02 00:02:00+00:00    ICRF position [    8.315    -7.480    30.011] ...
2026-02-02 00:03:00+00:00    ICRF position [    8.641    -9.549    29.539] ...
2026-02-02 00:04:00+00:00    ICRF position [    8.933   -11.595    28.933] ...
                                                   ...                        
2026-02-02 23:56:00+00:00    ICRF position [    9.510   -14.759    27.068] ...
2026-02-02 23:57:00+00:00    ICRF position [    9.728   -16.655    26.121] ...
2026-02-02 23:58:00+00:00    ICRF position [    9.906   -18.497    25.049] ...
2026-02-02 23:59:00+00:00    ICRF position [   10.043   -20.275    23.858] ...
2026-02-03 00:00:00+00:00    ICRF position [   10.139   -21.980    22.551] ...
Freq: min, Name: -158755, Length: 1441, dtype: skyfield_position[<class 'skyfield.positionlib.ICRF'>]

In [6]:
nanoff_a_berlin_sr: pd.Series = psfe.at(nanoff_a - berlin_gs, times_idx)
nanoff_a_berlin_sr

2026-02-02 00:00:00+00:00    ICRF position [ 4511.502 -8967.640 -5810.344] ...
2026-02-02 00:01:00+00:00    ICRF position [ 4448.793 -8902.835 -6256.888] ...
2026-02-02 00:02:00+00:00    ICRF position [ 4380.571 -8808.970 -6698.170] ...
2026-02-02 00:03:00+00:00    ICRF position [ 4307.174 -8686.406 -7132.277] ...
2026-02-02 00:04:00+00:00    ICRF position [ 4228.966 -8535.628 -7557.328] ...
                                                   ...                        
2026-02-02 23:56:00+00:00    ICRF position [ 4127.596 -8268.187 -8282.754] ...
2026-02-02 23:57:00+00:00    ICRF position [ 4034.865 -8044.678 -8673.218] ...
2026-02-02 23:58:00+00:00    ICRF position [ 3938.423 -7795.916 -9047.972] ...
2026-02-02 23:59:00+00:00    ICRF position [ 3838.730 -7522.930 -9405.405] ...
2026-02-03 00:00:00+00:00    ICRF position [ 3736.259 -7226.851 -9743.980] ...
Freq: min, Name: -158810, Length: 1441, dtype: skyfield_position[<class 'skyfield.positionlib.ICRF'>]

Note that both `nanoff_rel_sr` and `nanoff_a_berlin_sr` now are using the `ICRF` frame in the background instead of `Geocentric` as they are describing relative coordinates whose centrum is NanoFF-A or the ground station, respectively, and not the Earth center as in the case of `nanoff_a_sr`.

### Manual creation of a position and Series

It is also possible to create a Skyfield position `Series` manually if the position has been already created somehow differently.
As an example the `nanoff_a_pos` Skyfield position object can be used directly with `psfe.sf_position_to_series` to create a `Series` of Skyfield positions. This has the disadvantage that the `DatetimeIndex` is not automatically set and the users is responsible for of the alignment of the position and time. 

In [7]:
psfe.sf_position_to_series(nanoff_a_pos, index=times_idx)

2026-02-02 00:00:00+00:00    Geocentric position [ 1330.651 -6747.060  -764...
2026-02-02 00:01:00+00:00    Geocentric position [ 1258.258 -6696.250 -1210...
2026-02-02 00:02:00+00:00    Geocentric position [ 1180.413 -6616.421 -1652...
2026-02-02 00:03:00+00:00    Geocentric position [ 1097.455 -6507.936 -2086...
2026-02-02 00:04:00+00:00    Geocentric position [ 1009.747 -6371.279 -2511...
                                                   ...                        
2026-02-02 23:56:00+00:00    Geocentric position [  947.411 -6046.651 -3236...
2026-02-02 23:57:00+00:00    Geocentric position [  844.991 -5837.133 -3627...
2026-02-02 23:58:00+00:00    Geocentric position [  738.922 -5602.405 -4001...
2026-02-02 23:59:00+00:00    Geocentric position [  629.663 -5343.495 -4359...
2026-02-03 00:00:00+00:00    Geocentric position [  517.688 -5061.533 -4697...
Freq: min, Length: 1441, dtype: skyfield_position[<class 'skyfield.positionlib.Geocentric'>]

It is also possible to create a `SkyfieldPositionExtensionArray` manually, although that is not recommended:

In [8]:
psfe.SkyfieldPositionExtensionArray(nanoff_a_pos)

<SkyfieldPositionExtensionArray>
[Geocentric position [ 1330.651 -6747.060  -764.348] km and velocity [   -1.159     0.604    -7.475] km/s at time 2026-02-02T00:00:00+00:00 with center=399 target=-158810,
 Geocentric position [ 1258.258 -6696.250 -1210.867] km and velocity [   -1.253     1.089    -7.403] km/s at time 2026-02-02T00:01:00+00:00 with center=399 target=-158810,
 Geocentric position [ 1180.413 -6616.421 -1652.124] km and velocity [   -1.341     1.570    -7.300] km/s at time 2026-02-02T00:02:00+00:00 with center=399 target=-158810,
 Geocentric position [ 1097.455 -6507.936 -2086.207] km and velocity [   -1.423     2.044    -7.164] km/s at time 2026-02-02T00:03:00+00:00 with center=399 target=-158810,
 Geocentric position [ 1009.747 -6371.279 -2511.233] km and velocity [   -1.499     2.509    -6.998] km/s at time 2026-02-02T00:04:00+00:00 with center=399 target=-158810,
 Geocentric position [  917.670 -6207.057 -2925.365] km and velocity [   -1.569     2.963    -6.801] km/s a

## Using the Skyfield position Series

The created pandas `Series` can now be used as any other `Series` too. Especially sliced and iterated and added to a `DataFrame`.

### Accessing and slicing

Slicing with the index, here `DatetimeIndex`:

In [9]:
# Accessing a specific time point, here with a string
nanoff_a_sr.loc["2026-02-02T12:00:00Z"]

<Geocentric ICRS position and velocity at date t center=399 target=-158810>

In [10]:
# Accessing a time interval, here with a datetime object
start_time_slice = datetime.datetime.fromisoformat("2026-02-02T12:00:00Z")
end_time_slice = datetime.datetime.fromisoformat("2026-02-02T12:05:00Z")
nanoff_a_sr.loc[start_time_slice:end_time_slice]

2026-02-02 12:00:00+00:00    Geocentric position [ -988.107  6218.345  2830...
2026-02-02 12:01:00+00:00    Geocentric position [ -890.798  6031.835  3235...
2026-02-02 12:02:00+00:00    Geocentric position [ -789.603  5819.020  3626...
2026-02-02 12:03:00+00:00    Geocentric position [ -684.965  5580.829  4000...
2026-02-02 12:04:00+00:00    Geocentric position [ -577.341  5318.301  4358...
2026-02-02 12:05:00+00:00    Geocentric position [ -467.199  5032.583  4696...
Freq: min, Name: NanoFF-A, dtype: skyfield_position[<class 'skyfield.positionlib.Geocentric'>]

Slicing with integers:

In [11]:
# Accessing by a singular integer position
nanoff_a_sr.iloc[100]

<Geocentric ICRS position and velocity at date t center=399 target=-158810>

In [12]:
# Accessing by a slice of integer positions with a step of 2
nanoff_a_sr.iloc[slice(30, 40, 2)]

2026-02-02 00:30:00+00:00    Geocentric position [-1496.041  3140.142 -5984...
2026-02-02 00:32:00+00:00    Geocentric position [-1589.970  3898.664 -5492...
2026-02-02 00:34:00+00:00    Geocentric position [-1656.487  4589.969 -4905...
2026-02-02 00:36:00+00:00    Geocentric position [-1694.405  5202.028 -4234...
2026-02-02 00:38:00+00:00    Geocentric position [-1703.026  5724.141 -3489...
Freq: 2min, Name: NanoFF-A, dtype: skyfield_position[<class 'skyfield.positionlib.Geocentric'>]

Slicing with `boolean` mask will be discussed later in the ["Slicing with the `skyfield_position` `Series` Accessor"](#slicing-with-the-skyfield_position-series-accessor) section

### Arithmetic operations

While it is possible to create relative positions by creating first a `VectorSum` and then using the result in `psfe.at()` this is not always desirable. For example when it is needed to use multiple TLEs to describe a satellite, then each `EarthSatellite` can only hold one TLE and different satellites might need to change TLEs at different times. It can therefore be usable to first calculate the position on each TLE for a given time period (e.g. from TLE epoch to following TLE epoch), then concatenate all `Series` for a satellite and only afterwards calculate the relative position between two satellites or a satellite and a ground station.

In [13]:
# Define a second satellite Series
nanoff_b_sr: pd.Series = psfe.at(nanoff_b, times_idx)
nanoff_b_sr

2026-02-02 00:00:00+00:00    Geocentric position [ 1338.216 -6750.367  -733...
2026-02-02 00:01:00+00:00    Geocentric position [ 1266.214 -6701.646 -1180...
2026-02-02 00:02:00+00:00    Geocentric position [ 1188.728 -6623.901 -1622...
2026-02-02 00:03:00+00:00    Geocentric position [ 1106.096 -6517.485 -2056...
2026-02-02 00:04:00+00:00    Geocentric position [ 1018.680 -6382.874 -2482...
                                                   ...                        
2026-02-02 23:56:00+00:00    Geocentric position [  956.921 -6061.410 -3209...
2026-02-02 23:57:00+00:00    Geocentric position [  854.718 -5853.789 -3601...
2026-02-02 23:58:00+00:00    Geocentric position [  748.827 -5620.902 -3976...
2026-02-02 23:59:00+00:00    Geocentric position [  639.706 -5363.769 -4335...
2026-02-03 00:00:00+00:00    Geocentric position [  527.827 -5083.513 -4675...
Freq: min, Name: NanoFF-B, Length: 1441, dtype: skyfield_position[<class 'skyfield.positionlib.Geocentric'>]

In [14]:
# Relative position between the two position Series
nanoff_b_sr - nanoff_a_sr

2026-02-02 00:00:00+00:00    ICRF position [    7.565    -3.307    30.551] ...
2026-02-02 00:01:00+00:00    ICRF position [    7.955    -5.396    30.348] ...
2026-02-02 00:02:00+00:00    ICRF position [    8.315    -7.480    30.011] ...
2026-02-02 00:03:00+00:00    ICRF position [    8.641    -9.549    29.539] ...
2026-02-02 00:04:00+00:00    ICRF position [    8.933   -11.595    28.933] ...
                                                   ...                        
2026-02-02 23:56:00+00:00    ICRF position [    9.510   -14.759    27.068] ...
2026-02-02 23:57:00+00:00    ICRF position [    9.728   -16.655    26.121] ...
2026-02-02 23:58:00+00:00    ICRF position [    9.906   -18.497    25.049] ...
2026-02-02 23:59:00+00:00    ICRF position [   10.043   -20.275    23.858] ...
2026-02-03 00:00:00+00:00    ICRF position [   10.139   -21.980    22.551] ...
Freq: min, Length: 1441, dtype: skyfield_position[<class 'skyfield.positionlib.ICRF'>]

Skyfield currently only supports the `__sub__` operator for positions, so that is the only arithmetic operation possible at the moment. Also `__eq__` is not supported so comparisons on the position themselves are not possible.

## The `skyfield_position` `Series` Accessor

The `skyfield_position` `Series` Accessor allows to access common derived data of Skyfield positions.

In [15]:
nanoff_rel_sr.skyfield_position

### Simple properties

In [16]:
# Accessing the underlying Skyfield position object
nanoff_a_sr.skyfield_position.position

<Geocentric ICRS position and velocity at date t center=399 target=-158810>

In [17]:
# Accessing the center of the positions frame, 399 for example is the center of the Earth
nanoff_a_sr.skyfield_position.center

399

In [18]:
# Similar to the center the target of the positions frame can be accessed, for satellites this is -1 followed by the NORAD catalog ID of the satellite, for example -158810 for NanoFF-A with NORAD ID 58810
nanoff_a_sr.skyfield_position.target

-158810

In [19]:
# Accessing the length component will return a Series of the Astropy Extension array type with the correct unit, here the length of the relative position vector, so the distance between the two satellites
nanoff_rel_sr.skyfield_position.length

2026-02-02 00:00:00+00:00     31.64660766480495 km
2026-02-02 00:01:00+00:00     31.83457280302519 km
2026-02-02 00:02:00+00:00    32.027539580897475 km
2026-02-02 00:03:00+00:00     32.22470180426538 km
2026-02-02 00:04:00+00:00    32.425252184998996 km
                                     ...          
2026-02-02 23:56:00+00:00     32.26381228334279 km
2026-02-02 23:57:00+00:00    32.470190177513885 km
2026-02-02 23:58:00+00:00    32.676076999477935 km
2026-02-02 23:59:00+00:00     32.88069467698172 km
2026-02-03 00:00:00+00:00     33.08327779402989 km
Freq: min, Name: length, Length: 1441, dtype: unit[km]

### More advanced properties and functions

In [20]:
# Accessing the x, y and z components of the position as a pandas DataFrame with Quantity UnitsDtype
nanoff_a_sr.skyfield_position.xyz

,x,y,z
2026-02-02 00:00:00+00:00,1330.6511750133932 km,-6747.060153182442 km,-764.3481875200815 km
2026-02-02 00:01:00+00:00,1258.258441042206 km,-6696.24976253448 km,-1210.8671184092202 km
2026-02-02 00:02:00+00:00,1180.413133140199 km,-6616.421443001309 km,-1652.1244746036548 km
2026-02-02 00:03:00+00:00,1097.4552478728413 km,-6507.935975320599 km,-2086.206541517927 km
2026-02-02 00:04:00+00:00,1009.7466918613346 km,-6371.278595921668 km,-2511.232980290524 km
...,...,...,...
2026-02-02 23:56:00+00:00,947.410783644335 km,-6046.651002549271 km,-3236.7607472869067 km
2026-02-02 23:57:00+00:00,844.9909167385426 km,-5837.133260295946 km,-3627.198743447495 km
2026-02-02 23:58:00+00:00,738.9216486739757 km,-5602.405196433733 km,-4001.92800851328 km
2026-02-02 23:59:00+00:00,629.6629679975924 km,-5343.494802121336 km,-4359.335814597282 km


In [21]:
nanoff_a_sr.skyfield_position.xyz.dtypes

x    unit[km]
y    unit[km]
z    unit[km]
dtype: object

Different properties may be combined with `pd.concat` to a `pd.DataFrame`:

In [22]:
# Combining the Series returned by the time, length and speed properties to a DataFrame
pd.concat([
    nanoff_a_sr.skyfield_position.time,
    nanoff_a_sr.skyfield_position.length,
    nanoff_a_sr.skyfield_position.speed,
], axis=1)

,time,length,speed
2026-02-02 00:00:00+00:00,2026-02-02 00:00:00+00:00,6919.370015542747 km,7.5884908392252335 km / s
2026-02-02 00:01:00+00:00,2026-02-02 00:01:00+00:00,6920.200457005684 km,7.5873853036346635 km / s
2026-02-02 00:02:00+00:00,2026-02-02 00:02:00+00:00,6920.977037664682 km,7.5862554891891705 km / s
2026-02-02 00:03:00+00:00,2026-02-02 00:03:00+00:00,6921.697509558468 km,7.585108700822072 km / s
2026-02-02 00:04:00+00:00,2026-02-02 00:04:00+00:00,6922.360176260213 km,7.5839529821126455 km / s
...,...,...,...
2026-02-02 23:56:00+00:00,2026-02-02 23:56:00+00:00,6923.597018513785 km,7.581669826142333 km / s
2026-02-02 23:57:00+00:00,2026-02-02 23:57:00+00:00,6924.067090394988 km,7.580567045144267 km / s
2026-02-02 23:58:00+00:00,2026-02-02 23:58:00+00:00,6924.476656992198 km,7.579491577697813 km / s
2026-02-02 23:59:00+00:00,2026-02-02 23:59:00+00:00,6924.826416452315 km,7.578453425494265 km / s


The most relevant position data can be retrieved via the `to_astropy_dataframe()` function:

In [23]:
nanoff_a_df: pd.DataFrame = nanoff_a_sr.skyfield_position.to_astropy_dataframe()
nanoff_a_df

,time,x,y,z,vx,vy,vz,length,speed
2026-02-02 00:00:00+00:00,2026-02-02 00:00:00+00:00,1330.6511750133932 km,-6747.060153182442 km,-764.3481875200815 km,-1.1593142248501966 km / s,0.6035419883019602 km / s,-7.475086675980539 km / s,6919.370015542747 km,7.5884908392252335 km / s
2026-02-02 00:01:00+00:00,2026-02-02 00:01:00+00:00,1258.258441042206 km,-6696.24976253448 km,-1210.8671184092202 km,-1.2528961729943071 km / s,1.0894823986775657 km / s,-7.403492083367096 km / s,6920.200457005684 km,7.5873853036346635 km / s
2026-02-02 00:02:00+00:00,2026-02-02 00:02:00+00:00,1180.413133140199 km,-6616.421443001309 km,-1652.1244746036548 km,-1.3410028375002463 km / s,1.5704584554992944 km / s,-7.2997701317657855 km / s,6920.977037664682 km,7.5862554891891705 km / s
2026-02-02 00:03:00+00:00,2026-02-02 00:03:00+00:00,1097.4552478728413 km,-6507.935975320599 km,-2086.206541517927 km,-1.4232557477786318 km / s,2.0443782766600576 km / s,-7.164407480149509 km / s,6921.697509558468 km,7.585108700822072 km / s
2026-02-02 00:04:00+00:00,2026-02-02 00:04:00+00:00,1009.7466918613346 km,-6371.278595921668 km,-2511.232980290524 km,-1.4993032297870654 km / s,2.5091865365032433 km / s,-6.998029407274324 km / s,6922.360176260213 km,7.5839529821126455 km / s
...,...,...,...,...,...,...,...,...,...
2026-02-02 23:56:00+00:00,2026-02-02 23:56:00+00:00,947.410783644335 km,-6046.651002549271 km,-3236.7607472869067 km,-1.674094505186255 km / s,3.276616760778375 km / s,-6.62894467795052 km / s,6923.597018513785 km,7.581669826142333 km / s
2026-02-02 23:57:00+00:00,2026-02-02 23:57:00+00:00,844.9909167385426 km,-5837.133260295946 km,-3627.198743447495 km,-1.7386692369456849 km / s,3.7047637425959743 km / s,-6.380967922029718 km / s,6924.067090394988 km,7.580567045144267 km / s
2026-02-02 23:58:00+00:00,2026-02-02 23:58:00+00:00,738.9216486739757 km,-5602.405196433733 km,-4001.92800851328 km,-1.7956986412802531 km / s,4.116662079783719 km / s,-6.1055099940109105 km / s,6924.476656992198 km,7.579491577697813 km / s
2026-02-02 23:59:00+00:00,2026-02-02 23:59:00+00:00,629.6629679975924 km,-5343.494802121336 km,-4359.335814597282 km,-1.8449456965957887 km / s,4.510554845960358 km / s,-5.803794162492498 km / s,6924.826416452315 km,7.578453425494265 km / s


In [24]:
# The dtypes of the created DataFrame show that the time column is of numpy datetime64 type, while the other Series are of UnitsDtype with physical type length or velocity
nanoff_a_df.dtypes

time      datetime64[us, UTC]
x                    unit[km]
y                    unit[km]
z                    unit[km]
vx               unit[km / s]
vy               unit[km / s]
vz               unit[km / s]
length               unit[km]
speed            unit[km / s]
dtype: object

For relative positions with a `GeographicPosition` as a center it is possible to calculate the altitude, azimuth and distance in the centers local horizon frame.

In [25]:
# Calculate the altitude, azimuth and distance of NanoFF A as seen from the Berlin ground station
nanoff_a_berlin_altaz_df: pd.DataFrame = nanoff_a_berlin_sr.skyfield_position.altaz()
nanoff_a_berlin_altaz_df

,altitude,azimuth,distance
2026-02-02 00:00:00+00:00,-59.350493681306034 deg,53.924204376331744 deg,11598.806470917963 km
2026-02-02 00:01:00+00:00,-60.849524723394204 deg,56.93876264121242 deg,11755.887092427929 km
2026-02-02 00:02:00+00:00,-62.30711385795643 deg,60.15528820576358 deg,11901.799847885108 km
2026-02-02 00:03:00+00:00,-63.71684417029363 deg,63.60292807178521 deg,12036.393808263932 km
2026-02-02 00:04:00+00:00,-65.07109294580003 deg,67.31340709385634 deg,12159.535650671489 km
...,...,...,...
2026-02-02 23:56:00+00:00,-68.04976048374998 deg,72.24199582043175 deg,12409.834599739284 km
2026-02-02 23:57:00+00:00,-69.2131285041537 deg,76.95692106559184 deg,12498.867251281814 km
2026-02-02 23:58:00+00:00,-70.27417708507735 deg,82.09283963918821 deg,12575.900621335015 km
2026-02-02 23:59:00+00:00,-71.21622343764251 deg,87.6724042892714 deg,12640.884211807388 km


In [26]:
nanoff_a_berlin_altaz_df.dtypes

altitude    unit[deg]
azimuth     unit[deg]
distance     unit[km]
dtype: object

It is also possible to concatenate two `DataFrame` objects:

In [27]:
# First the position data and the altaz DataFrame are constructed through the accessor and then concatenated to one single DataFrame
pd.concat([
    nanoff_a_berlin_sr.skyfield_position.to_astropy_dataframe(),
    nanoff_a_berlin_sr.skyfield_position.altaz(),
], axis=1)

,time,x,y,z,vx,vy,vz,length,speed,altitude,azimuth,distance
2026-02-02 00:00:00+00:00,2026-02-02 00:00:00+00:00,4511.501795437433 km,-8967.640231676865 km,-5810.343516074134 km,-0.9974000156449803 km / s,0.8364304666417253 km / s,-7.475507326926107 km / s,11598.806470918003 km,7.587992653637786 km / s,-59.350493681306034 deg,53.924204376331744 deg,11598.806470917963 km
2026-02-02 00:01:00+00:00,2026-02-02 00:01:00+00:00,4448.793313678783 km,-8902.83532543549 km,-6256.887608888262 km,-1.0920024568020847 km / s,1.3230770662908407 km / s,-7.403910157140424 km / s,11755.887092427973 km,7.600058414513253 km / s,-60.849524723394204 deg,56.93876264121242 deg,11755.887092427929 km
2026-02-02 00:02:00+00:00,2026-02-02 00:02:00+00:00,4380.570935910707 km,-8808.970253035272 km,-6698.169972142627 km,-1.1811326943011897 km / s,1.8047548406953062 km / s,-7.3001856203636 km / s,11901.799847885153 km,7.612156367831905 km / s,-62.30711385795643 deg,60.15528820576358 deg,11901.799847885108 km
2026-02-02 00:03:00+00:00,2026-02-02 00:03:00+00:00,4307.174474487385 km,-8686.406063918754 km,-7132.276890773049 km,-1.2644122379587066 km / s,2.2793718943150894 km / s,-7.1648203756180235 km / s,12036.393808263978 km,7.62423278467837 km / s,-63.71684417029363 deg,63.60292807178521 deg,12036.393808263932 km
2026-02-02 00:04:00+00:00,2026-02-02 00:04:00+00:00,4228.965652996248 km,-8535.628264023384 km,-7557.328025442276 km,-1.3414893940794568 km / s,2.744872888146499 km / s,-6.998439701709388 km / s,12159.535650671538 km,7.636234623488958 km / s,-65.07109294580003 deg,67.31340709385634 deg,12159.535650671489 km
...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-02 23:56:00+00:00,2026-02-02 23:56:00+00:00,4127.596212274374 km,-8268.187479737433 km,-8282.754378901913 km,-1.512110685673034 km / s,3.5094568270234623 km / s,-6.629365629338823 km / s,12409.83459973933 km,7.651879154425211 km / s,-68.04976048374998 deg,72.24199582043175 deg,12409.834599739284 km
2026-02-02 23:57:00+00:00,2026-02-02 23:57:00+00:00,4034.864780992839 km,-8044.678117310916 km,-8673.217554813713 km,-1.5777056992971081 km / s,3.938310303143157 km / s,-6.381386296761421 km / s,12498.867251281863 km,7.662997741480324 km / s,-69.2131285041537 deg,76.95692106559184 deg,12498.867251281814 km
2026-02-02 23:58:00+00:00,2026-02-02 23:58:00+00:00,3938.4226391027823 km,-7795.916177672211 km,-9047.971844795216 km,-1.6357584668130918 km / s,4.350910663863142 km / s,-6.105925784077113 km / s,12575.900621335062 km,7.673881615409925 km / s,-70.27417708507735 deg,82.09283963918821 deg,12575.900621335015 km
2026-02-02 23:59:00+00:00,2026-02-02 23:59:00+00:00,3838.729590859471 km,-7522.9299206299465 km,-9405.404520480393 km,-1.6860319470366139 km / s,4.745500969363359 km / s,-5.804207359933775 km / s,12640.884211807434 km,7.684484774776124 km / s,-71.21622343764251 deg,87.6724042892714 deg,12640.884211807388 km


### Comparison operations and `boolean` arrays

The resulting `Series` of a property can be used for example in comparison operations to create a boolean mask. In the following example the relative distance between the two NanoFF satellites is compared with the value of `30 km` (represented by the astropy `Quantity` object `30 * u.km`) to calculate if they are in close proximity:

In [28]:
# Create a boolean Series for when the two satellites are in close proximity, here defined as the distance between the two of them being less than 30 km
close_proximity_sr: pd.Series = nanoff_rel_sr.skyfield_position.length < 30 * u.km
close_proximity_sr

2026-02-02 00:00:00+00:00    False
2026-02-02 00:01:00+00:00    False
2026-02-02 00:02:00+00:00    False
2026-02-02 00:03:00+00:00    False
2026-02-02 00:04:00+00:00    False
                             ...  
2026-02-02 23:56:00+00:00    False
2026-02-02 23:57:00+00:00    False
2026-02-02 23:58:00+00:00    False
2026-02-02 23:59:00+00:00    False
2026-02-03 00:00:00+00:00    False
Freq: min, Name: length, Length: 1441, dtype: bool

In [29]:
# Counting at how many timestamps the two satellites are in close proximity
close_proximity_sr.value_counts()

length
False    1206
True      235
Name: count, dtype: int64

### Slicing with the `skyfield_position` `Series` Accessor

It is possible to slice pandas `Series` and `DataFrame` with `boolean` arrays. For an example these can come from comparing one of our simple properties with a reference value or from another `Series` of the same `DataFrame`. In the following example the `close_proximity_sr` we created before in ["Comparison operations and `boolean` arrays"](#comparison-operations-and-boolean-arrays) is used to filter the `nanoff_rel_sr`.

In [30]:
# Indexing with the boolean Series to only get the positions when the two satellites are in close proximity
nanoff_rel_sr.loc[close_proximity_sr]

2026-02-02 01:15:00+00:00    ICRF position [   -3.858    28.267     9.248] ...
2026-02-02 01:16:00+00:00    ICRF position [   -3.290    27.593    11.139] ...
2026-02-02 01:17:00+00:00    ICRF position [   -2.709    26.801    12.978] ...
2026-02-02 01:18:00+00:00    ICRF position [   -2.117    25.894    14.757] ...
2026-02-02 01:19:00+00:00    ICRF position [   -1.516    24.875    16.470] ...
                                                   ...                        
2026-02-02 23:39:00+00:00    ICRF position [    1.370    17.646    23.549] ...
2026-02-02 23:40:00+00:00    ICRF position [    1.989    16.084    24.711] ...
2026-02-02 23:41:00+00:00    ICRF position [    2.602    14.445    25.770] ...
2026-02-02 23:42:00+00:00    ICRF position [    3.206    12.735    26.722] ...
2026-02-02 23:43:00+00:00    ICRF position [    3.799    10.959    27.562] ...
Name: -158755, Length: 235, dtype: skyfield_position[<class 'skyfield.positionlib.ICRF'>]

In [31]:
# Alternatively, we can directly index with the condition without creating the intermediate boolean Series
nanoff_rel_sr.loc[nanoff_rel_sr.skyfield_position.length < 30 * u.km]

2026-02-02 01:15:00+00:00    ICRF position [   -3.858    28.267     9.248] ...
2026-02-02 01:16:00+00:00    ICRF position [   -3.290    27.593    11.139] ...
2026-02-02 01:17:00+00:00    ICRF position [   -2.709    26.801    12.978] ...
2026-02-02 01:18:00+00:00    ICRF position [   -2.117    25.894    14.757] ...
2026-02-02 01:19:00+00:00    ICRF position [   -1.516    24.875    16.470] ...
                                                   ...                        
2026-02-02 23:39:00+00:00    ICRF position [    1.370    17.646    23.549] ...
2026-02-02 23:40:00+00:00    ICRF position [    1.989    16.084    24.711] ...
2026-02-02 23:41:00+00:00    ICRF position [    2.602    14.445    25.770] ...
2026-02-02 23:42:00+00:00    ICRF position [    3.206    12.735    26.722] ...
2026-02-02 23:43:00+00:00    ICRF position [    3.799    10.959    27.562] ...
Name: -158755, Length: 235, dtype: skyfield_position[<class 'skyfield.positionlib.ICRF'>]

This can be especially useful for the functions that return a `DataFrame`, like `altaz()`:

In [32]:
# Filter the altaz DataFrame for times when the satellite is above the horizon, here defined as an altitude greater than 0 degrees
nanoff_a_berlin_altaz_df.loc[nanoff_a_berlin_altaz_df["altitude"] > 0 * u.deg]

,altitude,azimuth,distance
2026-02-02 07:38:00+00:00,0.4007474292899744 deg,45.7158732806721 deg,2635.151648528335 km
2026-02-02 07:39:00+00:00,2.191859663596971 deg,55.0228860302884 deg,2445.712281273559 km
2026-02-02 07:40:00+00:00,3.491579724680588 deg,65.60724201817655 deg,2317.287497821974 km
2026-02-02 07:41:00+00:00,4.081523147819187 deg,77.0695656793679 deg,2261.048076596637 km
2026-02-02 07:42:00+00:00,3.8382374682925935 deg,88.70929497660721 deg,2282.7426179957283 km
...,...,...,...
2026-02-02 21:40:00+00:00,16.67388929659888 deg,284.32067658343306 deg,1401.130402364099 km
2026-02-02 21:41:00+00:00,13.559391455092513 deg,301.42667795153807 deg,1560.6459931637266 km
2026-02-02 21:42:00+00:00,9.548330587573542 deg,314.2507991650055 deg,1811.5416401123593 km
2026-02-02 21:43:00+00:00,5.578572862152168 deg,323.4968855717454 deg,2120.636778872508 km


### Properties not directly supported by `skyfield_position`

Any property or function on a Skyfield position object that is not natively supported by the `skyfield_position` `Series` Accessor is dispatched to the position itself. Thereby a warning is raised to let the user know that directly accessing the `position` property is favoured. If there is a need for additional properties feel free to create an Issue or PR. Note is is possible that this option will be removed in the future.

In [33]:
nanoff_a_sr.skyfield_position.light_time

/home/julian/studium/ilr/git/github/pandas-skyfield-extension/pandas_skyfield_extension/position_extension.py:586: UserWarning: Attribute light_time not found in SkyfieldPositionSeriesAccessor, delegating to position object. Consider using the position property directly instead.
  warnings.warn(msg, stacklevel=find_stack_level())


array([2.67135810e-07, 2.67167871e-07, 2.67197852e-07, ...,
       2.67332962e-07, 2.67346465e-07, 2.67357700e-07], shape=(1441,))

## Converting to other reference frames

It is possible to convert Skyfield positions to other reference frames, for example the True of Date frame (TOD). Possible frames can be found in `skyfield.framelib`.

In [34]:
# Convert the position Series to the TETE/TOD frame implemented by framelib.true_equator_and_equinox_of_date and return the position and velocity as a DataFrame
nanoff_a_sr.skyfield_position.frame_xyz_and_velocity(framelib.true_equator_and_equinox_of_date)

,x,y,z,vx,vy,vz
2026-02-02 00:00:00+00:00,1372.1542476509167 km,-6739.1003892512845 km,-761.1833394384379 km,-1.1437787036432632 km / s,0.5970965965517773 km / s,-7.477997001101549 km / s
2026-02-02 00:01:00+00:00,1300.603004558678 km,-6688.693694476251 km,-1207.8836248364967 km,-1.2403920216866862 km / s,1.0824761440616746 km / s,-7.406624654718556 km / s
2026-02-02 00:02:00+00:00,1223.4156526774166 km,-6609.301815054205 km,-1649.3352643373341 km,-1.3315829312299874 km / s,1.562921970516944 km / s,-7.3031112656583135 km / s
2026-02-02 00:03:00+00:00,1140.9294204319106 km,-6501.283624668565 km,-2083.6236951611454 km,-1.416959456859251 km / s,2.036344473912189 km / s,-7.167942597738304 km / s
2026-02-02 00:04:00+00:00,1053.5042610503563 km,-6365.122320384728 km,-2508.867678527899 km,-1.4961562803010418 km / s,2.5006904576522824 km / s,-7.0017431011726 km / s
...,...,...,...,...,...,...
2026-02-02 23:56:00+00:00,991.1187668744326 km,-6040.828739430469 km,-3234.5428433695074 km,-1.6763871172847808 km / s,3.2670653755469474 km / s,-6.63307827615062 km / s
2026-02-02 23:57:00+00:00,888.4669731130185 km,-5831.896244357381 km,-3625.2334690373077 km,-1.7441046426616662 km / s,3.694813892257044 km / s,-6.3852522286091355 km / s
2026-02-02 23:58:00+00:00,781.9778973235329 km,-5597.776047674184 km,-4000.223851412841 km,-1.8042517130480535 km / s,4.106356957007166 km / s,-6.109926421324105 km / s
2026-02-02 23:59:00+00:00,672.1134390624649 km,-5339.493503641104 km,-4357.900130521761 km,-1.8565778853396473 km / s,4.499939124610784 km / s,-5.808323575081795 km / s


## Interoperability with other libraries

### GeoPandas

GeoPandas is a library extending pandas with geospatial operations, it is not a dependency of this Pandas Skyfield Extension, but if installed it allows to convert a Skyfield position `Series` to a `GeoDataFrame`.

In [35]:
import geopandas as gpd
import folium
nanoff_a_gdf: gpd.GeoDataFrame = nanoff_a_sr.skyfield_position.to_geodataframe()
nanoff_a_gdf

,time,x,y,z,vx,vy,vz,length,speed,geometry
2026-02-02 00:00:00+00:00,2026-02-02 00:00:00+00:00,1330.6511750133932 km,-6747.060153182442 km,-764.3481875200815 km,-1.1593142248501966 km / s,0.6035419883019602 km / s,-7.475086675980539 km / s,6919.370015542747 km,7.5884908392252335 km / s,POINT (149.30499 -6.35465)
2026-02-02 00:01:00+00:00,2026-02-02 00:01:00+00:00,1258.258441042206 km,-6696.24976253448 km,-1210.8671184092202 km,-1.2528961729943071 km / s,1.0894823986775657 km / s,-7.403492083367096 km / s,6920.200457005684 km,7.5873853036346635 km / s,POINT (148.54932 -10.11328)
2026-02-02 00:02:00+00:00,2026-02-02 00:02:00+00:00,1180.413133140199 km,-6616.421443001309 km,-1652.1244746036548 km,-1.3410028375002463 km / s,1.5704584554992944 km / s,-7.2997701317657855 km / s,6920.977037664682 km,7.5862554891891705 km / s,POINT (147.78194 -13.86907)
2026-02-02 00:03:00+00:00,2026-02-02 00:03:00+00:00,1097.4552478728413 km,-6507.935975320599 km,-2086.206541517927 km,-1.4232557477786318 km / s,2.0443782766600576 km / s,-7.164407480149509 km / s,6921.697509558468 km,7.585108700822072 km / s,POINT (146.99786 -17.62135)
2026-02-02 00:04:00+00:00,2026-02-02 00:04:00+00:00,1009.7466918613346 km,-6371.278595921668 km,-2511.232980290524 km,-1.4993032297870654 km / s,2.5091865365032433 km / s,-6.998029407274324 km / s,6922.360176260213 km,7.5839529821126455 km / s,POINT (146.19147 -21.36945)
...,...,...,...,...,...,...,...,...,...,...
2026-02-02 23:56:00+00:00,2026-02-02 23:56:00+00:00,947.410783644335 km,-6046.651002549271 km,-3236.7607472869067 km,-1.674094505186255 km / s,3.276616760778375 km / s,-6.62894467795052 km / s,6923.597018513785 km,7.581669826142333 km / s,POINT (147.13087 -27.99775)
2026-02-02 23:57:00+00:00,2026-02-02 23:57:00+00:00,844.9909167385426 km,-5837.133260295946 km,-3627.198743447495 km,-1.7386692369456849 km / s,3.7047637425959743 km / s,-6.380967922029718 km / s,6924.067090394988 km,7.580567045144267 km / s,POINT (146.22486 -31.73021)
2026-02-02 23:58:00+00:00,2026-02-02 23:58:00+00:00,738.9216486739757 km,-5602.405196433733 km,-4001.92800851328 km,-1.7956986412802531 km / s,4.116662079783719 km / s,-6.1055099940109105 km / s,6924.476656992198 km,7.579491577697813 km / s,POINT (145.26443 -35.45559)
2026-02-02 23:59:00+00:00,2026-02-02 23:59:00+00:00,629.6629679975924 km,-5343.494802121336 km,-4359.335814597282 km,-1.8449456965957887 km / s,4.510554845960358 km / s,-5.803794162492498 km / s,6924.826416452315 km,7.578453425494265 km / s,POINT (144.23573 -39.17286)


In [36]:
nanoff_a_gdf.dtypes

time        datetime64[us, UTC]
x                      unit[km]
y                      unit[km]
z                      unit[km]
vx                 unit[km / s]
vy                 unit[km / s]
vz                 unit[km / s]
length                 unit[km]
speed              unit[km / s]
geometry               geometry
dtype: object

The `EPSG:4326` `crs` corresponds to the WGS 84 Ellipsoid also used by the `skyfield_position.subpoint` attribute, it is therefore used in the background to create the `GeoDataFrame`:

In [37]:
nanoff_a_gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

Using `GeoPandas` it is possible to plot data from our position `DataFrame` in a interactive map using the `explore()` function, here the positions of NanoFF A coloured in by the speed value:

In [38]:
# Currently it is not possible to plot astropy Quantity objects, therefore the speed Series is converted to a native python scalar
nanoff_a_gdf["speed_native"] = nanoff_a_gdf["speed"].array.to_quantity().to_value()

# Create a title of the map
title: str = f"NanoFF-A positions and respective velocities in km/s between {nanoff_a_gdf.index.min()} and {nanoff_a_gdf.index.min()}"

# Define Figure properties
f: folium.Figure = folium.Figure(width="800", height="500", title=title)

# Reduce the frame to the plotable columns and use GeoDataFrame.explorer() to create a folium map
m: folium.map = nanoff_a_gdf[["geometry", "time", "speed_native"]].explore(column="speed_native", legend=True)

# Add the map to the figure
f.add_child(m)
f